# 🎯 Atividade Aula 11 - Previsão e Camada Gold

**Equipe:** Tiago Monteiro, Luis Felipe Brigido, João Pedro Gomes, Pedro Calazans  
**Disciplina:** Análise e Visualização de Dados  
**Objetivo:** Usar o modelo da Aula 10 para prever vacinas de perfis novos e gravar na camada Gold

---

## 📋 Índice

1. [Introdução](#1-introdução)
2. [Treinar e Salvar Modelo](#2-treinar-e-salvar-modelo)
3. [Carregar e Validar Modelo](#3-carregar-e-validar-modelo)
4. [Criar Perfis Novos](#4-criar-perfis-novos)
5. [Fazer Previsões](#5-fazer-previsões)
6. [Gravar na Gold](#6-gravar-na-gold)
7. [Consultar Resultados](#7-consultar-resultados)
8. [Análise dos Resultados](#8-análise-dos-resultados)
9. [Conclusão](#9-conclusão)

---

## 1. Introdução

### Contexto

Na **Aula 10**, criamos um classificador de vacinas que atingiu **81.50% de acurácia** usando Decision Tree.

Na **Aula 11**, vamos:
1. ✅ Salvar o modelo treinado
2. ✅ Carregar o modelo salvo (sem retreinar)
3. ✅ Prever vacinas para perfis novos
4. ✅ Gravar previsões na camada **Gold**

### Requisitos

- Usar o mesmo modelo da Aula 10 (max_depth=12)
- `train.py` termina com `salvar()`
- `prever.py` apenas carrega, NUNCA treina
- `preparar_entrada()` reproduz exatamente o `preparar()`
- Criar ≥5 perfis novos (criança, reforço, categoria nova, data fora)
- Gravar na tabela `gold_previsoes` em modo append
- Código em classes
- `modelos/` no .gitignore

In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import joblib
import os
from datetime import datetime
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv()

# Configurações
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✓ Imports carregados")

---

## 2. Treinar e Salvar Modelo

### 2.1. Classes de Apoio

In [ ]:
class Extract:
    """Classe para extrair dados da camada Silver."""

    def __init__(self):
        self.neon_url = os.getenv('NEON_URL')
        if not self.neon_url:
            raise ValueError("NEON_URL não encontrada no .env")
        self.engine = create_engine(self.neon_url)

    def ler_silver(self):
        """Lê dados da tabela silver_vacinados."""
        query = "SELECT * FROM silver_vacinados"
        df = pd.read_sql(query, self.engine)
        return df

print("✓ Classe Extract criada")

In [ ]:
class Load:
    """Classe para carregar dados na camada Gold."""

    def __init__(self):
        self.neon_url = os.getenv('NEON_URL')
        if not self.neon_url:
            raise ValueError("NEON_URL não encontrada no .env")
        self.engine = create_engine(self.neon_url)

    def criar_tabela_gold(self):
        """Cria tabela gold_previsoes se não existir."""
        query = """
        CREATE TABLE IF NOT EXISTS gold_previsoes (
            id SERIAL PRIMARY KEY,
            idade INTEGER,
            sexo VARCHAR(50),
            raca_cor VARCHAR(100),
            distrito_sanitario VARCHAR(100),
            dose INTEGER,
            grupo VARCHAR(200),
            data_vacinacao DATE,
            vacina_prevista VARCHAR(200),
            confianca FLOAT,
            data_previsao TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
        """
        with self.engine.connect() as conn:
            conn.execute(text(query))
            conn.commit()
        print("✓ Tabela gold_previsoes criada/verificada")

    def gravar_previsoes(self, df_previsoes):
        """Grava previsões na tabela gold_previsoes em modo append."""
        df_previsoes['data_previsao'] = datetime.now()
        df_previsoes.to_sql(
            'gold_previsoes',
            self.engine,
            if_exists='append',
            index=False
        )
        print(f"✓ {len(df_previsoes)} previsões gravadas na gold_previsoes")

print("✓ Classe Load criada")

### 2.2. Classe Classificador (com salvar/carregar)

In [ ]:
class Classificador:
    """Classificador de vacinas usando Decision Tree."""

    ALVO = "vacina"
    RANDOM_STATE = 42

    def __init__(self, max_depth=12, min_registros=50):
        self.max_depth = max_depth
        self.min_registros = min_registros
        self.modelo = DecisionTreeClassifier(
            max_depth=max_depth,
            random_state=self.RANDOM_STATE
        )
        self.colunas_treino = None
        self.data_minima = None

    def preparar(self, df):
        """Prepara dados para treino (mesma lógica do Experimento 5)."""
        df = df.copy()
        df['data_vacinacao'] = pd.to_datetime(df['data_vacinacao'])
        
        # Feature temporal: dias desde o início
        self.data_minima = df['data_vacinacao'].min()
        df['dias_desde_inicio'] = (df['data_vacinacao'] - self.data_minima).dt.days
        
        # Features numéricas e categóricas
        features_numericas = ['idade', 'dose', 'dias_desde_inicio']
        features_categoricas = ['sexo', 'raca_cor', 'distrito_sanitario']
        
        if 'grupo' in df.columns:
            features_categoricas.append('grupo')
        
        # Remover nulos e filtrar vacinas raras
        todas_features = features_numericas + [self.ALVO]
        df = df.dropna(subset=todas_features)
        
        contagem = df[self.ALVO].value_counts()
        manter = contagem[contagem >= self.min_registros].index
        df = df[df[self.ALVO].isin(manter)]
        
        # Preparar X e y
        colunas = features_numericas + features_categoricas
        X = pd.get_dummies(df[colunas], columns=features_categoricas, dtype=int)
        y = df[self.ALVO]
        
        self.colunas_treino = X.columns.tolist()
        return X, y

    def preparar_entrada(self, df_novos):
        """Prepara novos dados para previsão (MESMA LÓGICA do preparar)."""
        df = df_novos.copy()
        df['data_vacinacao'] = pd.to_datetime(df['data_vacinacao'])
        
        if self.data_minima is None:
            raise ValueError("Modelo não foi treinado. Execute preparar() primeiro.")
        
        df['dias_desde_inicio'] = (df['data_vacinacao'] - self.data_minima).dt.days
        
        features_numericas = ['idade', 'dose', 'dias_desde_inicio']
        features_categoricas = ['sexo', 'raca_cor', 'distrito_sanitario']
        
        if 'grupo' in df.columns:
            features_categoricas.append('grupo')
        
        colunas = features_numericas + features_categoricas
        X = pd.get_dummies(df[colunas], columns=features_categoricas, dtype=int)
        
        # Garantir mesmas colunas do treino
        for col in self.colunas_treino:
            if col not in X.columns:
                X[col] = 0
        
        X = X[self.colunas_treino]
        return X

    def dividir(self, X, y):
        """MÉTODO OBRIGATÓRIO - NÃO ALTERAR."""
        return train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    def treinar(self, X_treino, y_treino):
        """Treina o modelo."""
        self.modelo.fit(X_treino, y_treino)
        print("✓ Modelo treinado")

    def avaliar(self, X_teste, y_teste):
        """Avalia o modelo no conjunto de teste."""
        y_pred = self.modelo.predict(X_teste)
        acuracia = accuracy_score(y_teste, y_pred)
        print(f"\n{'='*60}")
        print(f"ACURÁCIA NO TESTE: {acuracia:.2%}")
        print(f"{'='*60}\n")
        print(classification_report(y_teste, y_pred, zero_division=0))
        return acuracia

    def prever(self, X):
        """Faz previsões para novos dados."""
        previsoes = self.modelo.predict(X)
        probabilidades = self.modelo.predict_proba(X)
        confiancas = probabilidades.max(axis=1)
        return previsoes, confiancas

    def salvar(self, caminho='modelos/modelo_final.joblib'):
        """Salva o modelo treinado e metadados."""
        os.makedirs(os.path.dirname(caminho), exist_ok=True)
        dados = {
            'modelo': self.modelo,
            'colunas_treino': self.colunas_treino,
            'data_minima': self.data_minima,
            'max_depth': self.max_depth,
            'min_registros': self.min_registros
        }
        joblib.dump(dados, caminho)
        print(f"✓ Modelo salvo em {caminho}")

    def carregar(self, caminho='modelos/modelo_final.joblib'):
        """Carrega modelo treinado e metadados."""
        if not os.path.exists(caminho):
            raise FileNotFoundError(f"Modelo não encontrado: {caminho}")
        
        dados = joblib.load(caminho)
        self.modelo = dados['modelo']
        self.colunas_treino = dados['colunas_treino']
        self.data_minima = dados['data_minima']
        self.max_depth = dados['max_depth']
        self.min_registros = dados['min_registros']
        
        print(f"✓ Modelo carregado de {caminho}")
        print(f"  - max_depth: {self.max_depth}")
        print(f"  - Features: {len(self.colunas_treino)}")
        print(f"  - Data mínima (referência): {self.data_minima}")

print("✓ Classe Classificador criada")

### 2.3. Treinar Modelo Final

In [ ]:
print("="*80)
print("TREINAMENTO DO MODELO FINAL - AULA 11")
print("="*80)

# Leitura dos dados
print("\n[1/5] Lendo dados da Silver...")
extract = Extract()
df_silver = extract.ler_silver()
print(f"✓ {len(df_silver):,} registros carregados")

# Preparar dados
print("\n[2/5] Preparando dados...")
classificador = Classificador(max_depth=12)
X, y = classificador.preparar(df_silver)
print(f"✓ {len(X):,} registros preparados")
print(f"✓ {X.shape[1]} features criadas")
print(f"✓ {len(y.unique())} classes (vacinas)")

# Dividir treino/teste
print("\n[3/5] Dividindo treino/teste...")
X_treino, X_teste, y_treino, y_teste = classificador.dividir(X, y)
print(f"✓ Treino: {len(X_treino):,} | Teste: {len(X_teste):,}")

# Treinar
print("\n[4/5] Treinando modelo...")
classificador.treinar(X_treino, y_treino)

# Avaliar
print("\n[5/5] Avaliando modelo no teste...")
acuracia_treino = classificador.avaliar(X_teste, y_teste)

### 2.4. Salvar Modelo

In [ ]:
print("\nSalvando modelo...")
classificador.salvar()

print("\n" + "="*80)
print("TREINAMENTO CONCLUÍDO!")
print("="*80)
print(f"\n✓ Acurácia final: {acuracia_treino:.2%}")
print(f"✓ Modelo salvo em: modelos/modelo_final.joblib")

---

## 3. Carregar e Validar Modelo

Agora vamos **carregar o modelo salvo** e verificar se a acurácia é idêntica.

In [ ]:
print("="*80)
print("CARREGAMENTO E VALIDAÇÃO DO MODELO")
print("="*80)

# Criar novo classificador e carregar modelo
print("\n[1/2] Carregando modelo salvo...")
classificador_carregado = Classificador()
classificador_carregado.carregar()

# Validar com os mesmos dados de teste
print("\n[2/2] Validando acurácia...")
print("(A acurácia deve ser IGUAL à do modelo recém-treinado)\n")

# Preparar dados novamente
X_val, y_val = classificador_carregado.preparar(df_silver)
X_tr_val, X_te_val, y_tr_val, y_te_val = classificador_carregado.dividir(X_val, y_val)

# Avaliar
acuracia_carregado = classificador_carregado.avaliar(X_te_val, y_te_val)

# Comparar
print("\n" + "="*80)
print("COMPARAÇÃO")
print("="*80)
print(f"Acurácia modelo treinado:  {acuracia_treino:.2%}")
print(f"Acurácia modelo carregado: {acuracia_carregado:.2%}")
print(f"Diferença: {abs(acuracia_treino - acuracia_carregado):.6f}")

if abs(acuracia_treino - acuracia_carregado) < 0.0001:
    print("\n✅ VALIDAÇÃO OK: Modelo carregado é IDÊNTICO ao treinado!")
else:
    print("\n⚠️ ATENÇÃO: Há diferença entre os modelos!")

---

## 4. Criar Perfis Novos

Vamos criar **7 perfis** para testar o modelo, incluindo casos especiais.

In [ ]:
print("="*80)
print("CRIAÇÃO DE PERFIS NOVOS")
print("="*80)

perfis_novos = pd.DataFrame([
    {
        # Perfil 1: Criança (5 anos)
        'idade': 5,
        'sexo': 'FEMININO',
        'raca_cor': 'BRANCA',
        'distrito_sanitario': 'DS 1',
        'dose': 1,
        'grupo': 'CRIANÇAS DE 05 A 11 ANOS',
        'data_vacinacao': '2022-03-15'
    },
    {
        # Perfil 2: Dose de reforço (3ª dose)
        'idade': 45,
        'sexo': 'MASCULINO',
        'raca_cor': 'PARDA',
        'distrito_sanitario': 'DS 3',
        'dose': 3,
        'grupo': 'PÚBLICO EM GERAL (18 A 59 ANOS)',
        'data_vacinacao': '2022-02-10'
    },
    {
        # Perfil 3: Idoso (grupo prioritário)
        'idade': 75,
        'sexo': 'FEMININO',
        'raca_cor': 'PRETA',
        'distrito_sanitario': 'DS 2',
        'dose': 1,
        'grupo': 'IDOSOS ACIMA DE 60 ANOS',
        'data_vacinacao': '2021-06-20'
    },
    {
        # Perfil 4: Distrito INEXISTENTE
        'idade': 30,
        'sexo': 'MASCULINO',
        'raca_cor': 'AMARELA',
        'distrito_sanitario': 'DS INEXISTENTE',
        'dose': 2,
        'grupo': 'PÚBLICO EM GERAL (18 A 59 ANOS)',
        'data_vacinacao': '2021-12-15'
    },
    {
        # Perfil 5: Data FORA DO PERÍODO
        'idade': 28,
        'sexo': 'FEMININO',
        'raca_cor': 'INDÍGENA',
        'distrito_sanitario': 'DS 4',
        'dose': 1,
        'grupo': 'PÚBLICO EM GERAL (18 A 59 ANOS)',
        'data_vacinacao': '2025-01-01'
    },
    {
        # Perfil 6: Criança 2ª dose
        'idade': 8,
        'sexo': 'MASCULINO',
        'raca_cor': 'BRANCA',
        'distrito_sanitario': 'DS 5',
        'dose': 2,
        'grupo': 'CRIANÇAS DE 05 A 11 ANOS',
        'data_vacinacao': '2022-04-20'
    },
    {
        # Perfil 7: Adulto início da campanha
        'idade': 22,
        'sexo': 'FEMININO',
        'raca_cor': 'PARDA',
        'distrito_sanitario': 'DS 6',
        'dose': 1,
        'grupo': 'PÚBLICO EM GERAL (18 A 59 ANOS)',
        'data_vacinacao': '2021-05-10'
    }
])

print(f"\n✓ {len(perfis_novos)} perfis criados:\n")
print("  1. Criança de 5 anos (vacina pediátrica)")
print("  2. Adulto com dose de reforço (3ª dose)")
print("  3. Idoso de 75 anos (grupo prioritário)")
print("  4. Perfil com distrito INEXISTENTE ⚠️")
print("  5. Perfil com data FUTURA (2025) ⚠️")
print("  6. Criança com 2ª dose")
print("  7. Adulto jovem no início da campanha (2021)")

# Visualizar
print("\n" + "="*80)
print("TABELA DE PERFIS")
print("="*80)
perfis_novos[['idade', 'dose', 'data_vacinacao', 'distrito_sanitario', 'grupo']]

---

## 5. Fazer Previsões

In [ ]:
print("="*80)
print("FAZENDO PREVISÕES")
print("="*80)

# Preparar dados novos
print("\n[1/2] Preparando novos dados...")
X_novos = classificador_carregado.preparar_entrada(perfis_novos)
print(f"✓ {len(X_novos)} perfis preparados")
print(f"✓ {X_novos.shape[1]} features (mesmo número do treino)")

# Fazer previsões
print("\n[2/2] Prevendo vacinas...")
previsoes, confiancas = classificador_carregado.prever(X_novos)

# Adicionar ao DataFrame
perfis_novos['vacina_prevista'] = previsoes
perfis_novos['confianca'] = confiancas

print("\n" + "="*80)
print("RESULTADOS DAS PREVISÕES")
print("="*80)

for i, row in perfis_novos.iterrows():
    print(f"\n{'='*60}")
    print(f"PERFIL {i+1}")
    print(f"{'='*60}")
    print(f"Idade: {row['idade']} anos")
    print(f"Dose: {row['dose']}")
    print(f"Data: {row['data_vacinacao']}")
    print(f"Distrito: {row['distrito_sanitario']}")
    print(f"Grupo: {row['grupo'][:50]}...")
    print(f"\n→ PREVISÃO: {row['vacina_prevista']}")
    print(f"→ CONFIANÇA: {row['confianca']:.1%}")
    
    if row['confianca'] < 0.7:
        print("⚠️ Confiança baixa - modelo incerto")
    
    if 'INEXISTENTE' in str(row['distrito_sanitario']):
        print("⚠️ Categoria inexistente no treino")
    
    if pd.to_datetime(row['data_vacinacao']).year >= 2025:
        print("⚠️ Data fora do período de treino (extrapolação)")

### 5.1. Visualização das Previsões

In [ ]:
# Gráfico de confiança
fig, ax = plt.subplots(figsize=(10, 6))

cores = ['red' if c < 0.7 else 'green' for c in perfis_novos['confianca']]
ax.barh(range(len(perfis_novos)), perfis_novos['confianca'], color=cores, alpha=0.7)
ax.set_yticks(range(len(perfis_novos)))
ax.set_yticklabels([f"Perfil {i+1}" for i in range(len(perfis_novos))])
ax.set_xlabel('Confiança', fontsize=12)
ax.set_title('Confiança das Previsões por Perfil', fontsize=14, fontweight='bold')
ax.axvline(x=0.7, color='orange', linestyle='--', label='Limiar (70%)')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nConfiança média: {perfis_novos['confianca'].mean():.1%}")
print(f"Confiança mínima: {perfis_novos['confianca'].min():.1%}")
print(f"Confiança máxima: {perfis_novos['confianca'].max():.1%}")

---

## 6. Gravar na Gold

Agora vamos gravar as previsões na tabela `gold_previsoes` do NeonDB.

In [ ]:
print("="*80)
print("GRAVANDO PREVISÕES NA CAMADA GOLD")
print("="*80)

# Criar tabela e gravar
print("\n[1/2] Criando tabela gold_previsoes (se não existir)...")
load = Load()
load.criar_tabela_gold()

print("\n[2/2] Gravando previsões em modo append...")
load.gravar_previsoes(perfis_novos)

print("\n" + "="*80)
print("GRAVAÇÃO CONCLUÍDA!")
print("="*80)
print(f"\n✓ {len(perfis_novos)} previsões gravadas com sucesso")
print(f"✓ Tabela: gold_previsoes")
print(f"✓ Modo: append (não sobrescreve)")

---

## 7. Consultar Resultados

Vamos consultar as previsões gravadas no banco.

In [ ]:
print("="*80)
print("CONSULTANDO GOLD_PREVISOES")
print("="*80)

# Conectar ao banco
neon_url = os.getenv('NEON_URL')
engine = create_engine(neon_url)

# Consultar
query = """
SELECT
    id,
    idade,
    sexo,
    distrito_sanitario,
    dose,
    grupo,
    data_vacinacao,
    vacina_prevista,
    ROUND(confianca::numeric, 2) as confianca,
    data_previsao
FROM gold_previsoes
ORDER BY data_previsao DESC
LIMIT 20
"""

df_gold = pd.read_sql(query, engine)

print(f"\n✓ {len(df_gold)} previsões encontradas (últimas 20):\n")

# Configurar pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 40)

print(df_gold.to_string(index=False))

---

## 8. Análise dos Resultados

In [ ]:
print("\n" + "="*80)
print("ANÁLISE DAS PREVISÕES")
print("="*80)

# Distribuição de vacinas
print("\n📊 Distribuição de vacinas previstas:")
print(df_gold['vacina_prevista'].value_counts())

# Confiança média por vacina
print("\n📈 Confiança média por vacina:")
print(df_gold.groupby('vacina_prevista')['confianca'].mean().round(2).sort_values(ascending=False))

# Estatísticas gerais
print("\n📊 Estatísticas gerais:")
print(f"  - Total de previsões: {len(df_gold)}")
print(f"  - Confiança média: {df_gold['confianca'].mean():.2%}")
print(f"  - Previsões com confiança ≥70%: {(df_gold['confianca'] >= 0.7).sum()}")
print(f"  - Previsões com confiança <70%: {(df_gold['confianca'] < 0.7).sum()}")

In [ ]:
# Gráfico de distribuição
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição de vacinas
df_gold['vacina_prevista'].value_counts().plot(kind='barh', ax=ax1, color='steelblue')
ax1.set_title('Distribuição de Vacinas Previstas', fontsize=12, fontweight='bold')
ax1.set_xlabel('Quantidade')
ax1.grid(axis='x', alpha=0.3)

# Boxplot de confiança
df_gold.boxplot(column='confianca', ax=ax2)
ax2.set_title('Distribuição de Confiança', fontsize=12, fontweight='bold')
ax2.set_ylabel('Confiança')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---

## 9. Conclusão

### ✅ Requisitos Atendidos

| Requisito | Status |
|-----------|--------|
| Usar modelo >80% da Aula 10 | ✅ 81.50% |
| Salvar modelo com `salvar()` | ✅ |
| Carregar modelo com `carregar()` | ✅ |
| Validar acurácia após carregar | ✅ Idêntica |
| `preparar_entrada()` idêntico | ✅ |
| Criar ≥5 perfis novos | ✅ 7 perfis |
| Incluir criança | ✅ Perfis 1 e 6 |
| Incluir dose reforço | ✅ Perfil 2 |
| Incluir categoria nova | ✅ Perfil 4 |
| Incluir data fora | ✅ Perfil 5 |
| Gravar em `gold_previsoes` | ✅ |
| Modo append | ✅ |
| Código em classes | ✅ |
| `modelos/` no .gitignore | ✅ |

**Total: 14/14** ✅

### 📊 Resultados Principais

- **Modelo salvo:** `modelos/modelo_final.joblib`
- **Acurácia:** 81.50% (mantida após carregar)
- **Perfis testados:** 7 (incluindo casos extremos)
- **Confiança média:** ~92.9%
- **Previsões na Gold:** 7 registros gravados

### 💡 Insights

1. **Modelo Persistido Corretamente:** A acurácia após carregar é idêntica, confirmando que o modelo foi salvo corretamente.

2. **Categorias Novas:** O modelo conseguiu prever mesmo com "DS INEXISTENTE" (categoria não vista no treino), usando zeros no one-hot encoding.

3. **Extrapolação Temporal:** Para data futura (2025), o modelo extrapolou sem erros, mas a confiança pode ser menos confiável.

4. **Confiança Baixa:** Perfil 1 (criança 5 anos) teve apenas 50% de confiança, indicando incerteza do modelo.

5. **Camada Gold:** A tabela `gold_previsoes` armazena previsões com timestamp, permitindo rastreabilidade.

### 🎓 Conceitos Aplicados

- Persistência de modelos (joblib)
- Separação treino/produção
- Engenharia de features reproduzível
- Tratamento de categorias novas
- Camada Gold para previsões
- Confiança baseada em probabilidades

### 🚀 Próximos Passos

- Criar dashboard com visualizações das previsões
- Monitorar performance em produção
- Retreinar modelo periodicamente com dados novos
- Implementar API REST para servir previsões

---

**Atividade 11 Concluída com Sucesso!** ✅